# 01 — Data Exploration
Explore the Wikipedia pageview time-series stored in the SQLite database.

**Run `collect_data.py` (or start `main.py`) before executing this notebook.**

In [ ]:
import sys; sys.path.insert(0, '..')  # project root

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from backend.data.data_loader import get_available_pages, get_series, get_all_series
from backend.utils.io_utils import init_db

init_db()

## Available series

In [ ]:
pages = get_available_pages()
print(f"{len(pages)} series in database")
pages[:20]

## Single series — views over time

In [ ]:
PAGE = 'Python'

df = get_series(PAGE)
print(f"{PAGE}: {len(df)} days  |  mean={df.views.mean():.0f}  std={df.views.std():.0f}")

fig = px.line(df.reset_index(), x='date', y='views',
              title=f'Daily Page Views — {PAGE}',
              template='plotly_dark')
fig.show()

## Distribution of mean daily views across all pages

In [ ]:
all_df = get_all_series()

stats = (
    all_df.groupby('series_id')['views']
    .agg(mean='mean', std='std', total='sum', n='count')
    .sort_values('mean', ascending=False)
    .reset_index()
)
stats.head(20)

In [ ]:
fig = px.histogram(
    stats, x='mean', nbins=50, log_x=True,
    title='Distribution of Mean Daily Views (log scale)',
    template='plotly_dark', labels={'mean': 'Mean daily views'},
)
fig.show()

## Top-10 pages by average views

In [ ]:
top10 = stats.head(10).copy()

fig = px.bar(
    top10, x='series_id', y='mean', error_y='std',
    title='Top 10 Wikipedia Pages by Mean Daily Views',
    template='plotly_dark',
    labels={'series_id': 'Page', 'mean': 'Mean daily views'},
)
fig.update_xaxes(tickangle=-30)
fig.show()

## Multi-series overlay (top 5)

In [ ]:
top5 = stats.head(5)['series_id'].tolist()

fig = go.Figure()
for page in top5:
    s = get_series(page)
    fig.add_trace(go.Scatter(x=s.index, y=s['views'], name=page, mode='lines'))

fig.update_layout(
    title='Top 5 Pages — Daily Views Overlay',
    template='plotly_dark',
    hovermode='x unified',
)
fig.show()

## Seasonality decomposition (single series)

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

s = get_series(PAGE)['views'].dropna()
dec = seasonal_decompose(s, model='additive', period=7, extrapolate_trend='freq')

components = {
    'Observed':  dec.observed,
    'Trend':     dec.trend,
    'Seasonal':  dec.seasonal,
    'Residual':  dec.resid,
}

from plotly.subplots import make_subplots
fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=list(components.keys()))

for i, (name, comp) in enumerate(components.items(), 1):
    fig.add_trace(go.Scatter(x=comp.index, y=comp.values, name=name,
                             line=dict(width=1.2)), row=i, col=1)

fig.update_layout(height=800, template='plotly_dark',
                  title=f'Seasonal Decomposition — {PAGE} (period=7)',
                  showlegend=False)
fig.show()